# Exercices · Classification (facultatif)

[![Ouvrir dans Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josephazar/ELC2-Introduction-IA-2026/blob/main/seance-2/ia-elc2/03-exercice-classification.ipynb)

**ELC2 · Introduction au machine learning · Séance 2**

Ces exercices sont **facultatifs**. Ils reprennent le notebook `03-classification`, avec les mêmes données : les clients de Comtoise Télécom.

**Comment ça marche.**

- La partie **« Préparation fournie »** est déjà écrite : exécutez ses cellules dans l'ordre (**Maj + Entrée**),
  sans rien changer.
- Chaque exercice a une **consigne**, un **indice**, souvent le résultat **à retrouver** pour vérifier,
  et des **cellules de code vides** : c'est à vous de les remplir.
- Les cellules **« Votre réponse »** sont du texte : double-cliquez dessus pour écrire.
- Le notebook `03-classification` est votre modèle : copiez, collez, adaptez.

---
# Préparation fournie

Rien à écrire ici : exécutez les cellules, dans l'ordre.

## 0. Vérifier que tout est prêt

In [1]:
import sys
from pathlib import Path

python_utilise = Path(sys.executable)

if "google.colab" in sys.modules:
    print("OK : vous êtes sur Google Colab, il n'y a rien à installer.")
elif ".venv" in python_utilise.parts:
    print("Python utilisé :", ".../" + "/".join(python_utilise.parts[-4:]))
    print("OK : ce notebook tourne dans l'environnement virtuel du cours.")
else:
    print("Python utilisé :", python_utilise)
    print("ATTENTION : ce n'est pas le Python du cours. Relisez le guide d'installation.")

Python utilisé : .../ia-elc2/.venv/bin/python
OK : ce notebook tourne dans l'environnement virtuel du cours.


In [2]:
from urllib.request import urlretrieve

fichier = Path("donnees/clients-comtoise-telecom.xlsx")
if not fichier.exists():
    fichier.parent.mkdir(exist_ok=True)
    urlretrieve("https://raw.githubusercontent.com/josephazar/ELC2-Introduction-IA-2026/main/seance-2/ia-elc2/donnees/clients-comtoise-telecom.xlsx", fichier)
    print("Fichier téléchargé depuis GitHub.")

print("Données prêtes :", fichier)

Données prêtes : donnees/clients-comtoise-telecom.xlsx


## Les outils

In [3]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

# avertissement sans conséquence, que certaines versions de scikit-learn affichent avec la régression logistique
warnings.filterwarnings("ignore", message="Unknown solver options")

plt.rcParams.update({
    "figure.dpi": 100,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.alpha": 0.25,
    "axes.axisbelow": True,
})

## Les données, nettoyées comme dans le notebook 03

Les doublons sont retirés, la cible `y` est séparée des caractéristiques `X`, et les valeurs impossibles
(âges, mensualités et consommations absurdes) sont vidées.

In [4]:
clients = pd.read_excel("donnees/clients-comtoise-telecom.xlsx")
clients = clients.drop_duplicates().reset_index(drop=True)

y = (clients["resilie"] == "Oui").astype(int)
X = clients.drop(columns=["resilie", "id_client", "motif_resiliation"])

X["age"] = X["age"].mask((X["age"] < 18) | (X["age"] > 100))
X["mensualite"] = X["mensualite"].mask((X["mensualite"] < 5) | (X["mensualite"] > 200))
X["conso_go"] = X["conso_go"].mask(X["conso_go"] > 5000)

print(len(clients), "clients,", X.shape[1], "caractéristiques")

1500 clients, 9 caractéristiques


## Le découpage : 80 % pour le train, 20 % pour le test

`X_train`, `y_train` servent à apprendre ; `X_test`, `y_test` servent à juger.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print("train :", len(X_train), "clients     test :", len(X_test), "clients")

train : 1200 clients     test : 300 clients


## La chaîne de préparation

`fabriquer_preparation()` donne la même chaîne qu'au notebook 03 : elle remplit les cases vides, plafonne les
gros consommateurs, standardise les nombres, et encode le texte. Avec `echelle=False`, elle ne standardise pas.
Elle apprend sur le train, et applique ce qu'elle a appris au test.

In [6]:
ORDRE_CONTRAT = ["Mensuel", "1 an", "2 ans"]
ORDRE_SATISFACTION = ["Très insatisfait", "Insatisfait", "Neutre", "Satisfait", "Très satisfait"]

q1, q3 = X_train["conso_go"].quantile([0.25, 0.75])
borne_haute_conso = q3 + 1.5 * (q3 - q1)


def plafonner(valeurs):
    return np.minimum(valeurs, borne_haute_conso)


def traitement_nombres(echelle):
    etapes = [("vides", SimpleImputer(strategy="median"))]
    if echelle:
        etapes.append(("echelle", StandardScaler()))
    return etapes


def fabriquer_preparation(echelle=True):
    return ColumnTransformer([
        ("conso", Pipeline([("plafond", FunctionTransformer(plafonner, feature_names_out="one-to-one"))]
                           + traitement_nombres(echelle)), ["conso_go"]),
        ("nombres", Pipeline(traitement_nombres(echelle)),
         ["age", "anciennete_mois", "mensualite", "nb_appels_support"]),
        ("ordinales", Pipeline([
            ("vides", SimpleImputer(strategy="most_frequent")),
            ("codes", OrdinalEncoder(categories=[ORDRE_CONTRAT, ORDRE_SATISFACTION])),
        ]), ["contrat", "satisfaction"]),
        ("sans_ordre", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["ville", "mode_paiement"]),
    ]).set_output(transform="pandas")


fabriquer_preparation().fit(X_train).transform(X_test).shape

(300, 17)

La chaîne donne, pour le test, 300 lignes et 17 colonnes : tout est prêt.

---
# Les exercices

## Exercice 1 · Découper autrement

Dans le notebook, le test représentait 20 % des clients. **Découpez X et y une seconde fois, avec 30 %
pour le test.** Gardez `stratify=y` et `random_state=42`. Appelez les quatre résultats
`X_train_30`, `X_test_30`, `y_train_30`, `y_test_30`, pour ne pas écraser le découpage fourni.

Affichez ensuite :
- le nombre de clients dans le train et dans le test ;
- la part de clients qui résilient dans le train, puis dans le test.

**Indice.** La séance a utilisé `train_test_split(X, y, test_size=..., stratify=y, random_state=42)`.
La part de clients qui résilient est la **moyenne** d'une colonne de 0 et de 1 : `.mean()`.

**Pour vérifier.** 1 050 clients dans le train, 450 dans le test, et environ 26 % de résiliations dans chacun.

**Pourquoi garde-t-on `stratify=y` ?**

*Écrivez votre réponse ici (double-cliquez pour modifier).*

## Exercice 2 · Le modèle paresseux

Un modèle répond **« reste » (0) à tous les clients** du test, sans rien regarder. Sans utiliser
scikit-learn, répondez à deux questions, avec le test fourni (`y_test`) :

1. Quelle part des clients du test ce modèle classe-t-il correctement (son *accuracy*) ?
2. Combien de clients qui partent ce modèle trouve-t-il ? (son *recall*, en nombre de clients)

**Indice.** Fabriquez d'abord ses réponses : `np.zeros(len(y_test), dtype=int)` donne une liste de 0.
Comparer deux colonnes avec `==` donne des vrais et des faux, dont on peut prendre la `.mean()`
(la part des vrais) ou la `.sum()` (leur nombre). Un client est « trouvé » s'il part (`y_test == 1`)
**et** que le modèle l'a prédit (`predictions == 1`).

**Pour vérifier.** Une accuracy d'environ 74 %, et 0 client trouvé sur 79.

**Ce modèle est-il utile ? Pourquoi sa note paraît-elle bonne ?**

*Écrivez votre réponse ici (double-cliquez pour modifier).*

## Exercice 3 · La régression logistique et sa matrice de confusion

Avec la chaîne de préparation fournie, **entraînez une régression logistique** sur `X_train` et
`y_train`, puis :

1. affichez sa **matrice de confusion** sur le test ;
2. calculez **à la main**, à partir des quatre nombres VN, FP, FN, VP, la *precision* et le *recall* ;
3. comparez avec `precision_score` et `recall_score`.

**Indice.** Reprenez les sections 13 à 15 du notebook `03-classification`. Le modèle est un `Pipeline`
qui enchaîne `fabriquer_preparation()` et `LogisticRegression(max_iter=1000)`.
`confusion_matrix(y_test, predictions).ravel()` donne, dans l'ordre : VN, FP, FN, VP.
Les formules : precision = VP ÷ (VP + FP), recall = VP ÷ (VP + FN).

**Pour vérifier.** VN = 208, FP = 13, FN = 44, VP = 35. Precision d'environ 0,73, recall d'environ 0,44.

**Combien de clients qui partent le modèle a-t-il manqués ? Que cela représente-t-il pour l'entreprise ?**

*Écrivez votre réponse ici (double-cliquez pour modifier).*

## Exercice 4 · Le seuil

L'équipe commerciale ne peut appeler que **100 clients** parmi les 300 du test. Avec la régression
logistique de l'exercice 3 :

1. récupérez la **probabilité de résilier** de chaque client du test ;
2. pour les seuils 0,2 ; 0,3 ; 0,35 ; 0,4 ; 0,5 et 0,6, affichez le **nombre de clients contactés** (ceux dont
   la probabilité atteint le seuil) et le **recall** ;
3. choisissez le **seuil le plus bas** qui respecte la limite de 100 appels.

**Indice.** `regression.predict_proba(X_test)[:, 1]` donne la probabilité de résilier. Pour un seuil,
`(probabilites >= seuil).astype(int)` donne les prédictions : leur `.sum()` est le nombre de clients contactés.
Reprenez la section 16 du notebook `03-classification`.

**Pour vérifier.** À 0,3, il faudrait appeler 110 clients : c'est trop. Le bon seuil est entre 0,3 et 0,4.

**Quel seuil retenez-vous, et quel recall obtenez-vous ?**

*Écrivez votre réponse ici (double-cliquez pour modifier).*

## Exercice 5 · Choisir K : apprendre par cœur ou comprendre

KNN a un réglage : **K**, le nombre de voisins. Entraînez **trois KNN, avec K = 1, K = 5 et K = 25**, avec la
chaîne de préparation fournie. Pour chacun, affichez l'accuracy sur le **train** et sur le **test**.

**Indice.** Une boucle `for k in [1, 5, 25]:` et, pour chaque `k`, un `Pipeline` qui enchaîne
`fabriquer_preparation()` et `KNeighborsClassifier(n_neighbors=k)`. `accuracy_score(y_train, chaine.predict(X_train))`
donne l'accuracy sur le train. Reprenez les sections 17 et 19.

**Pour vérifier.** Pour K = 1, l'accuracy sur le train vaut 1,00.

**Quel K apprend par cœur ? Comment le voyez-vous ?**

*Écrivez votre réponse ici (double-cliquez pour modifier).*

## Exercice 6 · La fuite de données

Dans le fichier, la colonne `motif_resiliation` n'est remplie que pour les clients **partis**. Voyons ce que
donne un modèle qui la lit.

1. Fabriquez un tableau `motifs` d'une seule colonne : **1** si le motif est rempli, **0** sinon.
2. Entraînez un **arbre de décision de profondeur 1** uniquement avec cette colonne, sur les mêmes clients que
   `X_train`.
3. Calculez son accuracy sur les clients de `X_test`.

**Indice.** `clients[["motif_resiliation"]].notna().astype(int)` donne le tableau `motifs` (les doubles crochets
gardent un tableau à une colonne). `motifs.loc[X_train.index]` retrouve les mêmes clients que `X_train`.
L'arbre : `DecisionTreeClassifier(max_depth=1, random_state=42)`.

**Pour vérifier.** Une accuracy de 1,00 sur le test.

**Pourquoi ne peut-on pas utiliser ce modèle dans la vraie vie, malgré sa note parfaite ?**

*Écrivez votre réponse ici (double-cliquez pour modifier).*

## Exercice 7 (bonus) · Trois classes

La cible a maintenant **trois classes** (la cellule fournie ci-dessous la fabrique) :

- `reste` : le client reste ;
- `prix` : il part parce que c'est trop cher ;
- `autre` : il part pour une autre raison.

In [7]:
motif = clients["motif_resiliation"]
y3 = pd.Series(np.where(motif.isna(), "reste", np.where(motif == "Prix trop élevé", "prix", "autre")), index=clients.index)
y3_train, y3_test = y3.loc[X_train.index], y3.loc[X_test.index]

y3_test.value_counts()

reste    221
autre     58
prix      21
Name: count, dtype: int64

**Consigne.**

1. Entraînez une **régression logistique** (même préparation) pour prédire `y3_train`.
2. Affichez la **matrice de confusion 3 × 3** sur le test.
3. Lisez-la.

**Indice.** `ConfusionMatrixDisplay.from_predictions(y3_test, predictions)` fonctionne comme avec deux classes.
Vous pouvez aussi afficher `classification_report(y3_test, predictions, zero_division=0)` (le `zero_division=0` évite un
avertissement quand une classe n'est jamais prédite).

**Pour vérifier.** Une matrice de 3 lignes et 3 colonnes, dont la case « reste / reste » contient presque tous les
clients qui restent.

**Quelle classe le modèle retrouve-t-il le mieux ? Lesquelles confond-il ? Pourquoi, d'après les effectifs ?**

*Écrivez votre réponse ici (double-cliquez pour modifier).*

---
## Pour finir

Vous avez refait, avec vos mains, les gestes du notebook `03-classification` : **découper**, **mesurer** avec
une matrice de confusion, **régler** un seuil, **reconnaître** le sur-apprentissage et la fuite de données, et
**lire** une matrice à plus de deux classes.